In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from scipy.stats import gaussian_kde
from scipy.spatial import distance
from collections import Counter
import os
from tqdm import tqdm
from itertools import combinations


%matplotlib inline
plt.rcParams['figure.dpi'] = 150

In [2]:
# Distances among descriptors
path_to_pocketvec = '../processed/pocketvec_RUN/'

# Load tRNAs PocketVec descriptors
fps_tRNAs = pickle.load(open("../processed/pocketvec_RUN/fps_rank.pkl", 'rb'))
print(len(fps_tRNAs))

# Get the protein for each PocketVec descriptor
proteins = sorted(set([i.split("_")[1] for i in fps_tRNAs]))
print(len(proteins))

# Get outliers
outliers = {i: len(np.where(fps_tRNAs[i] > 128)[0]) for i in sorted(fps_tRNAs)}

# For each protein, get list of pockets
protein_to_pockets = {}
for p in sorted(set(proteins)):
    protein_to_pockets[p] = [i for i in fps_tRNAs if i.split("_")[1] == p and outliers[i] < 80]

print(len([j for i in sorted(protein_to_pockets) for j in protein_to_pockets[i]]))

276
21
264


In [3]:
def get_min_pocketvec_distance(fps1, fps2):
    """
    Get the minimum distance between two PocketVec descriptors
    """
    return np.min(pairwise_distances(fps1, fps2, metric='cosine').flatten())   

def get_min_pocketvec_distance_same_protein(fps1, fps2):
    """
    Get the minimum distance between two PocketVec descriptors,
    using only the upper triangle of the distance matrix.
    """
    dists = pairwise_distances(fps1, fps2, metric='cosine')
    triu_indices = np.triu_indices_from(dists, k=1)
    return np.min(dists[triu_indices])  

def binarize_distance(value, threshold=0.15):
    """
    Binaruze distance values
    """
    if value <= threshold:
        return 1
    else:
        return 0

In [14]:
# Load interpro annotations

pocket_detection_interpro_df = pd.read_csv("../processed/pocket_detection_data_interpro.tsv", sep="\t")

# Hide unnecessary columns
del pocket_detection_interpro_df['Pocket centroid coordinate (x y z)']
del pocket_detection_interpro_df['Pocket residues (chain_resn)']
del pocket_detection_interpro_df['B-factors']
del pocket_detection_interpro_df['Full path']
del pocket_detection_interpro_df['Interpro Matches']

# Add "pocket ID" column
pocket_detection_interpro_df['Pocket ID'] = ["_".join([j.strip(".pdb"),f"pocket_{k}"]) 
                                             for i,j,k in zip(pocket_detection_interpro_df['Uniprot AC'], 
                                             pocket_detection_interpro_df['File name'], 
                                             pocket_detection_interpro_df['Pocket number'])]

# Pocket to interpro
pocket_to_interpro = {}
for pocket in sorted(fps_tRNAs):
    annotations = pocket_detection_interpro_df[pocket_detection_interpro_df['Pocket ID'] == pocket]
    annotations = sorted(set(annotations["Interpro curated annotation"].tolist()))
    pocket_to_interpro[pocket] = annotations


# Load sequential comparisons

# NW algorithm
PROP_NW = pd.read_csv("../processed/sequences/NW_SeqAlign/Prop_matrix.tsv", sep='\t', index_col=0)
PROP_NW.columns = PROP_NW.columns.str.replace('(tRNA)', '')
PROP_NW.index = PROP_NW.index.str.replace('(tRNA)', '')
PROP_NW.columns = PROP_NW.columns.str.strip()
PROP_NW.index = PROP_NW.index.str.strip()
PROP_NW = PROP_NW.stack().to_dict()

SEQ_ID_NW = pd.read_csv("../processed/sequences/NW_SeqAlign/SeqId_matrix.tsv", sep='\t', index_col=0)
SEQ_ID_NW.columns = SEQ_ID_NW.columns.str.replace('(tRNA)', '')
SEQ_ID_NW.index = SEQ_ID_NW.index.str.replace('(tRNA)', '')
SEQ_ID_NW.columns = SEQ_ID_NW.columns.str.strip()
SEQ_ID_NW.index = SEQ_ID_NW.index.str.strip()
SEQ_ID_NW = SEQ_ID_NW.stack().to_dict()

# CLUSTAL OMEGA
with open('../processed/sequences/MSA_ClustalOmega/clustalo-I20250414-162824-0073-80384311-p1m.pim') as f:
    lines = f.readlines()
lines = [i for i in lines if i.startswith('#') == False and len(i.strip()) > 0]
clustal_omega = {}
for i, line in enumerate(lines):
    parts = line.split()
    uniprot_1 = parts[1].split('|')[0]
    seq_identities = list(map(float, parts[2:]))
    for j, identity in enumerate(seq_identities):
        uniprot_2 = lines[j].split()[1].split('|')[0]
        clustal_omega[(uniprot_1, uniprot_2)] = identity

# Load alphafill annotations
alphafill = pd.read_csv("../processed/alphafill_annotations.csv")
pocket_to_alphafill = {i + "_pocket_" + str(j): k for i,j,k in zip(alphafill['structure'], alphafill['pocket'], alphafill['distance'])}

# Load PDBe annotations
pdbe_annotations = pd.read_csv("../processed/pdbe_annotation_report.csv", sep='\t')
pdbe_annotations = pdbe_annotations.groupby('Pocket', as_index=False)['Min Distance'].min()
pocket_to_pdbe = {"_".join(i.split("_")[2:] + i.split("_")[:2]): j for i,j in zip(pdbe_annotations['Pocket'], pdbe_annotations['Min Distance'])}

# Load P2Rank scores
p2rank = pd.read_csv("../processed/pocket_detection_data.csv")
pocket_to_p2rank_score = {i.replace(".pdb", "") + "_pocket_" + str(j): k for i,j,k in zip(p2rank['File name'], p2rank['Pocket number'], p2rank['Pocket score'])}

# Load structural comparisons
PATH_TO_RMSDs = "../processed/structural_comparisons"
RMSDs = {}
for c, protein1 in enumerate(proteins):
    for protein2 in proteins[c+1:]:
        df = pd.read_csv(os.path.join(PATH_TO_RMSDs, f"{protein1}_{protein2}_rmsd.csv"))
        filename1 = [i.replace(".pdb", "") for i in df['file_name_1']]
        filename2 = [i.replace(".pdb", "") for i in df['file_name_2']]
        rmsds = [float(i) for i in df['rmsd']]
        for fn1, fn2, rmsd in zip(filename1, filename2, rmsds):
            RMSDs[tuple([fn1, fn2])] = rmsd

In [17]:
#############
### PAIRS ###
#############

In [25]:
# Prepare pd DF sumarizing ALL results (SEQ, STRUCTURE AND POCKETVEC)

ALL_RESULTS_PAIRS = []

# Load proteins
proteins = sorted(set(pd.read_csv(os.path.join("..", "processed", "pocket_detection_data.csv"))['Uniprot AC']))

# For each pair of proteins
for c,protein1 in tqdm(enumerate(proteins)):
    for protein2 in proteins[c+1:]:
        
        # Get all pockets from all structures
        pockets1 = protein_to_pockets[protein1]
        pockets2 = protein_to_pockets[protein2]

        # For each pair of pockets
        for pocket1 in pockets1:
            for pocket2 in pockets2:

                # Get interpro labels
                interpro_pocket1 = ";".join(pocket_to_interpro[pocket1])
                interpro_pocket2 = ";".join(pocket_to_interpro[pocket2])

                # Get pocketvec distance
                dist = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket2]), 3)

                # RMSD between structures
                rmsd = RMSDs[(pocket1.split("_pocket_")[0], pocket2.split("_pocket_")[0])]

                # Masif placeholder
                # ...

                # Distance to PDB ligand - NAN if not in dict
                dist_PDB_1 = pocket_to_pdbe.get(pocket1, np.nan)
                dist_PDB_2 = pocket_to_pdbe.get(pocket2, np.nan)

                # Distance to AlphaFill ligand - NAN if not in dict
                dist_alphafill_1 = pocket_to_alphafill.get(pocket1, np.nan)
                dist_alphafill_2 = pocket_to_alphafill.get(pocket2, np.nan)

                # P2Rank score
                p2rank_score_1 = pocket_to_p2rank_score[pocket1]
                p2rank_score_2 = pocket_to_p2rank_score[pocket2]

                results = [protein1, pocket1, interpro_pocket1, protein2, pocket2, interpro_pocket2, dist,
                           rmsd, SEQ_ID_NW[(protein1, protein2)], clustal_omega[(protein1, protein2)],
                           dist_PDB_1, dist_PDB_2, dist_alphafill_1, dist_alphafill_2, p2rank_score_1, p2rank_score_2]

                ALL_RESULTS_PAIRS.append(results)

# Store pandas df
ALL_RESULTS_PAIRS = pd.DataFrame(ALL_RESULTS_PAIRS, columns=["Protein1", "Pocket1", "InterPro-Pocket1", "Protein2", "Pocket2", "InterPro-Pocket2", 'PocketVec distance',
                                                 "Protein RMSD", 'Protein SEQ ID (NW)', 'Protein SEQ ID (CO)', 'Dist PDB 1', 'Dist PDB 2', 'Dist Alphafill 1', 'Dist AlphaFill 2',
                                                 "P2Rank score 1", "P2Rank score 2"])
ALL_RESULTS_PAIRS = ALL_RESULTS_PAIRS.sort_values('PocketVec distance').reset_index(drop=True)

0it [00:00, ?it/s]

21it [00:00, 56.68it/s]


In [29]:
len(set([tuple(sorted([i,j])) for i,j in zip(ALL_RESULTS_PAIRS['Protein1'], ALL_RESULTS_PAIRS['Protein2'])]))

210

In [43]:
len(ALL_RESULTS_PAIRS)

32561

In [38]:
ALL_RESULTS_PAIRS.to_csv("../processed/protein_prioritization/pairs.tsv", sep='\t', index=False)

In [31]:
################
### TRIPLETS ###
################

In [32]:
len(proteins) ** 3

9261

In [33]:
len([[i,j,k] for i in proteins for j in proteins for k in proteins])

9261

In [34]:
len([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins])

9261

In [35]:
len([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins if i != j and i != k and j != k])

7980

In [36]:
len(set([tuple(sorted([i,j,k])) for i in proteins for j in proteins for k in proteins if i != j and i != k and j != k]))

1330

In [37]:
ALL_RESULTS_TRIPLETS = []

# Load proteins
proteins = sorted(set(pd.read_csv(os.path.join("..", "processed", "pocket_detection_data.csv"))['Uniprot AC']))

# Calculate triplets
protein_triplets = list(combinations(proteins, 3))

# For each triplet of proteins
for protein1, protein2, protein3 in tqdm(protein_triplets):

    # Get all pockets from all structures
    pockets1 = protein_to_pockets[protein1]
    pockets2 = protein_to_pockets[protein2]
    pockets3 = protein_to_pockets[protein3]

    # For each pair of pockets
    for pocket1 in pockets1:
        for pocket2 in pockets2:
            for pocket3 in pockets3:

                # Get interpro labels
                interpro_pocket1 = ";".join(pocket_to_interpro[pocket1])
                interpro_pocket2 = ";".join(pocket_to_interpro[pocket2])
                interpro_pocket3 = ";".join(pocket_to_interpro[pocket3])

                # A (1-2), # B (1-3), #C (2-3)
                dist_A = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket2]), 3)
                dist_B = round(distance.cosine(fps_tRNAs[pocket1], fps_tRNAs[pocket3]), 3)
                dist_C = round(distance.cosine(fps_tRNAs[pocket2], fps_tRNAs[pocket3]), 3)

                # Masif placeholder
                # ...

                # RMSDs
                RMSD_A = RMSDs[(pocket1.split("_pocket_")[0], pocket2.split("_pocket_")[0])]
                RMSD_B = RMSDs[(pocket1.split("_pocket_")[0], pocket3.split("_pocket_")[0])]
                RMSD_C = RMSDs[(pocket2.split("_pocket_")[0], pocket3.split("_pocket_")[0])]

                # SEQ ID (CO)
                SEQ_ID_CO_A = clustal_omega[(protein1, protein2)]
                SEQ_ID_CO_B = clustal_omega[(protein1, protein3)]
                SEQ_ID_CO_C = clustal_omega[(protein2, protein3)]

                # SEQ ID (NW)
                SEQ_ID_NW_A = SEQ_ID_NW[(protein1, protein2)]
                SEQ_ID_NW_B = SEQ_ID_NW[(protein1, protein3)]
                SEQ_ID_NW_C = SEQ_ID_NW[(protein2, protein3)]

                # Distance to PDB ligand - NAN if not in dict
                dist_PDB_1 = pocket_to_pdbe.get(pocket1, np.nan)
                dist_PDB_2 = pocket_to_pdbe.get(pocket2, np.nan)
                dist_PDB_3 = pocket_to_pdbe.get(pocket3, np.nan)

                # Distance to AlphaFill ligand - NAN if not in dict
                dist_alphafill_1 = pocket_to_alphafill.get(pocket1, np.nan)
                dist_alphafill_2 = pocket_to_alphafill.get(pocket2, np.nan)
                dist_alphafill_3 = pocket_to_alphafill.get(pocket3, np.nan)

                # P2Rank score
                p2rank_score_1 = pocket_to_p2rank_score[pocket1]
                p2rank_score_2 = pocket_to_p2rank_score[pocket2]
                p2rank_score_3 = pocket_to_p2rank_score[pocket3]

                results = [protein1, pocket1, interpro_pocket1, 
                            protein2, pocket2, interpro_pocket2, 
                            protein3, pocket3, interpro_pocket3, 
                            dist_A, dist_B, dist_C,
                            RMSD_A, RMSD_B, RMSD_C,
                            SEQ_ID_CO_A, SEQ_ID_CO_B, SEQ_ID_CO_C,
                            SEQ_ID_NW_A, SEQ_ID_NW_B, SEQ_ID_NW_C,
                            dist_PDB_1, dist_PDB_2, dist_PDB_3,
                            dist_alphafill_1, dist_alphafill_2, dist_alphafill_3,
                            p2rank_score_1, p2rank_score_2, p2rank_score_3]

                ALL_RESULTS_TRIPLETS.append(results)


# Store pandas df
ALL_RESULTS_TRIPLETS = pd.DataFrame(ALL_RESULTS_TRIPLETS, columns=["protein1", "pocket1", "interpro_pocket1", 
                                                                    "protein2", "pocket2", "interpro_pocket2", 
                                                                    "protein3", "pocket3", "interpro_pocket3", 
                                                                    "dist_A", "dist_B", "dist_C",
                                                                    "RMSD_A", "RMSD_B", "RMSD_C",
                                                                    "SEQ_ID_CO_A", "SEQ_ID_CO_B", "SEQ_ID_CO_C",
                                                                    "SEQ_ID_NW_A", "SEQ_ID_NW_B", "SEQ_ID_NW_C",
                                                                    'Dist PDB 1', 'Dist PDB 2', 'Dist PDB 3',
                                                                    'Dist Alphafill 1', 'Dist AlphaFill 2', 'Dist Alphafill 3',
                                                                    "P2Rank score 1", "P2Rank score 2", "P2Rank score 3"])
# ALL_RESULTS_TRIPLETS = ALL_RESULTS_TRIPLETS.sort_values(["dist_A", "dist_B", "dist_C"]).reset_index(drop=True)

  0%|          | 0/1330 [00:00<?, ?it/s]

100%|██████████| 1330/1330 [01:18<00:00, 17.01it/s]


In [39]:
len(set([tuple(sorted([i,j,k])) for i,j,k in zip(ALL_RESULTS_TRIPLETS['protein1'], ALL_RESULTS_TRIPLETS['protein2'], ALL_RESULTS_TRIPLETS['protein3'])]))

1330

In [44]:
len(ALL_RESULTS_TRIPLETS)

2499258

In [41]:
ALL_RESULTS_TRIPLETS.to_csv("../processed/protein_prioritization/triplets.tsv", sep='\t', index=False)

In [46]:
Counter(ALL_RESULTS_TRIPLETS['Dist PDB 1'].isna())

Counter({True: 1919064, False: 580194})